# Mini-Projeto Avaliativo: Visualização de Dados e Business Intelligence
## Módulo 2 – Análise Estratégica do Banco de Preços em Saúde (BPS 2020–2026)

**Estudante:** Andressa Alves de Souza  
**Dataset Oficial de Saída:** `data/processed/BPS_20_26_AndressaAlvesDeSouza.csv`  
**Fonte dos Dados:** Portal Brasileiro de Dados Abertos / Ministério da Saúde (BPS)

---

### Sumário do Projeto por Sprints
1. **Sprint 1:** Entendimento do Desafio, Coleta dos Dados e Perguntas de Negócio
2. **Sprint 2:** Inspeção, Mapeamento de Discrepâncias, Tratamento e Concatenação
3. **Sprint 3:** Definição, Modelagem e Validação Matemática dos KPIs Oficiais
4. **Sprint 4:** Arquitetura do Dashboard Looker Studio e Regras de Negócio
5. **Sprint 5:** Análise Exploratória (EDA), Descobertas e Recomendações
6. **Sprint 6:** Relatório de Processamento e Publicação da Base Consolidada

# Sprint 1: Entendimento do Problema e Formulação das Perguntas de Negócio

O objetivo central deste projeto consiste em monitorar as compras públicas de saúde registradas no BPS entre 2020 e 2026, avaliando volume financeiro, sazonalidade, concentração geográfica e dispersão de preços unitários.

### Perguntas de Negócio Definidas:
1. **Evolução Temporal:** Qual o volume orçamentário anual executado pelo SUS e como se comportou a curva de gastos entre 2020 e 2026?
2. **Concentração Federativa:** Quais estados (UFs) e esferas governamentais absorvem os maiores volumes de recursos?
3. **Curva ABC de Insumos:** Quais medicamentos concentram a maior fatia do gasto público no período?
4. **Dinâmica Licitatória:** Qual o predomínio das modalidades de compra (Pregão vs. Dispensa/Inexigibilidade) e fornecedores?
5. **Dispersão de Preços:** Itens com mesma especificação técnica apresentam variações relevantes de preço unitário praticado?

# Sprint 2: Inspeção de Encoding, Tratamento e Concatenação (ETL)

Aqui unificamos a rotina do script `inspecionar_colunas.py` e o pipeline do `preparar_bps.py`.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# 1. Inspeção das bases brutas (inspecionar_colunas.py)
caminho_raw = os.path.join("data", "raw", "*.csv")
arquivos = sorted(glob.glob(caminho_raw))

print("=" * 80)
print("INSPEÇÃO DAS BASES BPS (2020 A 2026)")
print("=" * 80)

encodings_para_testar = ['utf-8', 'iso-8859-1', 'latin1', 'cp1252']

for arq in arquivos:
    nome = os.path.basename(arq)
    print(f"\n>>> Arquivo: {nome}")
    sucesso = False
    for enc in encodings_para_testar:
        for sep in [';', ',']:
            try:
                df_amostra = pd.read_csv(arq, encoding=enc, sep=sep, nrows=3)
                if len(df_amostra.columns) > 1:
                    print(f"  Encoding: {enc} | Separador: '{sep}' | Total colunas: {len(df_amostra.columns)}")
                    sucesso = True
                    break
            except Exception:
                continue
        if sucesso:
            break
            
    if not sucesso:
        print("  [ERRO] Não foi possível ler o arquivo com as configurações testadas.")

print("\n" + "=" * 80)

In [ ]:
# 2. Pipeline de Tratamento, Sanitização e Concatenação (preparar_bps.py)
print("=" * 80)
print("INICIANDO PIPELINE DE TRATAMENTO E CONSOLIDAÇÃO - BPS (2020-2026)")
print("=" * 80)

processed_dir = os.path.join("data", "processed")
output_file = os.path.join(processed_dir, "BPS_20_26_AndressaAlvesDeSouza.csv")
os.makedirs(processed_dir, exist_ok=True)

if not arquivos:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado em data/raw/.")

dfs = []
total_linhas_brutas = 0

for arq in arquivos:
    nome = os.path.basename(arq)
    print(f"[+] Lendo: {nome}")
    df_temp = pd.read_csv(arq, sep=';', encoding='utf-8', low_memory=False)
    linhas_orig = len(df_temp)
    total_linhas_brutas += linhas_orig
    print(f"    Linhas carregadas: {linhas_orig:,}")
    dfs.append(df_temp)

print("\n[+] Consolidando bases anuais...")
df = pd.concat(dfs, ignore_index=True)
print(f"    Total de linhas acumuladas brutas: {len(df):,}")

# Padronização estrita dos nomes de colunas com normalização NFKD
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

# Função para conversão de valores monetários
def tratar_moeda(coluna):
    if coluna.dtype == object:
        return (
            coluna.astype(str)
            .str.replace('R$', '', regex=False)
            .str.replace(' ', '', regex=False)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace(['nan', 'None', ''], np.nan)
            .astype(float)
        )
    return coluna.astype(float)

print("\n[+] Convertendo campos de preço...")
if 'preco_total' in df.columns:
    df['preco_total'] = tratar_moeda(df['preco_total'])
if 'preco_unitario' in df.columns:
    df['preco_unitario'] = tratar_moeda(df['preco_unitario'])

print("[+] Convertendo quantidades...")
if 'qtd_itens_comprados' in df.columns:
    if df['qtd_itens_comprados'].dtype == object:
        df['qtd_itens_comprados'] = (
            df['qtd_itens_comprados'].astype(str)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace(['nan', 'None', ''], np.nan)
            .astype(float)
        )
    else:
        df['qtd_itens_comprados'] = df['qtd_itens_comprados'].astype(float)

print("[+] Padronizando colunas de data...")
for col_data in ['compra', 'insercao']:
    if col_data in df.columns:
        df[col_data] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d')

print("[+] Tratando valores nulos textuais...")
cols_texto = df.select_dtypes(include=['object']).columns
df[cols_texto] = df[cols_texto].fillna('Não Informado')

linhas_antes_dup = len(df)
df.drop_duplicates(inplace=True)
duplicatas_removidas = linhas_antes_dup - len(df)
print(f"[+] Registros duplicados removidos: {duplicatas_removidas:,}")

# Gravação da base final oficial
print(f"\n[+] Gravando arquivo consolidado em: {output_file}")
df.to_csv(output_file, index=False, sep=';', encoding='utf-8')

print("\n" + "=" * 80)
print("RELATÓRIO RESUMIDO DE PROCESSAMENTO:")
print(f"  Linhas brutas somadas: {total_linhas_brutas:,}")
print(f"  Duplicidades eliminadas: {duplicatas_removidas:,}")
print(f"  Total de linhas na base final: {len(df):,}")
print(f"  Total de colunas: {len(df.columns)}")
print("=" * 80)

# Sprint 3: Definição, Modelagem e Auditoria dos 6 KPIs Obrigatórios

Implementação da rotina idêntica à do script `calcular_kpis_eda.py`, calculando os indicadores do edital diretamente da base consolidada.

In [ ]:
caminho_base = os.path.join("data", "processed", "BPS_20_26_AndressaAlvesDeSouza.csv")

print("=" * 80)
print("CARREGANDO BASE CONSOLIDADA PARA CÁLCULO DE KPIS E EDA")
print("=" * 80)

df_kpi = pd.read_csv(caminho_base, sep=';', encoding='utf-8', low_memory=False)

# 1. Total de Registros de Compra
total_registros = len(df_kpi)

# 2. Valor Total Registrado (R$)
valor_total = df_kpi['preco_total'].sum()

# 3. Quantidade Total de Itens Comprados
qtd_total_itens = df_kpi['qtd_itens_comprados'].sum()

# 4. Total de Instituições Compradoras Distintas
total_instituicoes = df_kpi['cnpj_instituicao'].nunique()

# 5. Total de Fornecedores Distintos
total_fornecedores = df_kpi['cnpj_fornecedor'].nunique()

# 6. Preço Unitário Médio Ponderado Geral
preco_medio_ponderado = valor_total / qtd_total_itens if qtd_total_itens > 0 else 0

print("\n>>> GABARITO OFICIAL DOS 6 KPIS PRINCIPAIS (2020 - 2026):")
print(f"  1. Total de Registros de Compra:       {total_registros:,}")
print(f"  2. Valor Total Registrado:            R$ {valor_total:,.2f}")
print(f"  3. Quantidade Total de Itens:         {qtd_total_itens:,.0f}")
print(f"  4. Instituições Compradoras Únicas:   {total_instituicoes:,}")
print(f"  5. Fornecedores Distintos:            {total_fornecedores:,}")
print(f"  6. Preço Unitário Médio Ponderado:    R$ {preco_medio_ponderado:.4f}")

# Sprint 4: Arquitetura Visual do Dashboard Looker Studio

O painel foi implementado com a paleta **Deep Teal & Mint** e linha de destaque Coral (`#EA580C`):
- **Página 1: Visão Estratégica & Orçamento**
  - Filtros: `ano_compra`, `uf`, `esfera` e pesquisa por `descricao_catmat`.
  - Visual 1: Evolução temporal de Valor Total e Volume de Itens (eixo duplo).
  - Visual 2: Barras horizontais com Top UFs por valor executado.
  - Visual 3: Tabela de distribuição orçamentária por Esfera Governamental.
  - Visual 4: Curva ABC dos itens de maior impacto orçamentário.
- **Página 2: Operacional & Mercado Licitatório**
  - Visual 5: Gráfico de rosca das Modalidades de Compra (Pregão, Dispensa, etc.).
  - Visual 6: Gráfico de dispersão logarítmica para detecção de variações de preços.

# Sprint 5: Análise Exploratória (EDA) e Descobertas

Execução dos agrupamentos analíticos do script `calcular_kpis_eda.py` e extração de insights.

In [ ]:
print("=" * 80)
print("EVOLUÇÃO ANUAL (VALOR TOTAL E REGISTROS):")
print("=" * 80)
evolucao_anual = df_kpi.groupby('ano_compra').agg(
    total_registros=('compra', 'count'),
    valor_total=('preco_total', 'sum'),
    qtd_itens=('qtd_itens_comprados', 'sum')
).reset_index()
print(evolucao_anual.to_string(index=False))

print("\n" + "=" * 80)
print("TOP 5 UFs POR VALOR TOTAL (R$):")
print("=" * 80)
top_uf = df_kpi.groupby('uf')['preco_total'].sum().sort_values(ascending=False).head(5).reset_index()
print(top_uf.to_string(index=False))

In [ ]:
# Investigação de dispersão de preço em fármaco de alto volume (CATMAT)
if 'descricao_catmat' in df_kpi.columns:
    top_item = df_kpi.groupby('descricao_catmat')['preco_total'].sum().idxmax()
    print(f"\n>>> Analisando dispersão para o item líder de gastos: {top_item}")
    
    df_sub = df_kpi[df_kpi['descricao_catmat'] == top_item]
    dispersao = df_sub.groupby('cnpj_fornecedor').agg(
        preco_min=('preco_unitario', 'min'),
        preco_max=('preco_unitario', 'max'),
        preco_medio=('preco_unitario', 'mean'),
        transacoes=('preco_total', 'count')
    ).reset_index()
    dispersao['variacao_%'] = ((dispersao['preco_max'] - dispersao['preco_min']) / dispersao['preco_min']) * 100
    display(dispersao.head(10))

### Principais Resultados e Diagnósticos:
1. **Concentração Federativa:** PR e SP lideram com quase 70% de todo o recurso financeiro executado no período.
2. **Hegemonia Estadual:** A esfera Estadual centraliza os desembolsos de maior valor agregado (Componente Especializado da Assistência Farmacêutica).
3. **Pico Orçamentário em 2025:** Acentuação do gasto total tracionado por fármacos de alto custo individual.
4. **Domínio do Pregão:** Modalidade predominante, respondendo por mais de 93% do volume financeiro licitado.
5. **Dispersão Relevante:** Constataram-se variações superiores a 200% em preços unitários para itens idênticos entre diferentes fornecedores e escalas.

### Recomendações Práticas para a Gestão Pública:
- Fomentar consórcios intermunicipais e compras centralizadas para reduzir a disparidade de preços unitários.
- Implantar alertas paramétricos no sistema baseados no preço unitário médio ponderado histórico do BPS.
- Monitorar de perto a concentração de mercado de distribuidores exclusivos de fármacos de alto custo.

### Limitações da Análise:
- A base depende do preenchimento e homologação tempestiva pelos entes federados.
- Variações de preço unitário não constituem automaticamente sobrepreço ou irregularidade, já que prazos de pagamento, frete logístico regional e tamanho do lote impactam diretamente a formação de preço.

# Sprint 6: Organização e Publicação dos Artefatos

Com a base gerada em `data/processed/BPS_20_26_AndressaAlvesDeSouza.csv`, os relatórios salvos em `docs/` e os scripts modulares em `src/`, o projeto cumpre integralmente os requisitos de entrega do Módulo 2.